# Nível 1 — Dados e primeira análise com LLM
Desafio Técnico — Estágio em Engenharia de IA

Este notebook cobre a Parte A (tratamento de dados e regras determinísticas em pandas)
e a Parte B (análise com LLM sobre um cliente sinalizado).

In [1]:
import json
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
with open("../dados/dados_nivel_1.json", encoding="utf-8") as f:
    dados = json.load(f)

taxa_cambio_usd_brl = dados["taxa_cambio_usd_brl"]
df = pd.DataFrame(dados["operacoes"])

print(f"Taxa de câmbio USD/BRL: {taxa_cambio_usd_brl}")
print(f"Total de operações carregadas: {len(df)}")
df.head(10)

Taxa de câmbio USD/BRL: 5.4
Total de operações carregadas: 20


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [3]:
print("=== Diagnóstico de qualidade dos dados ===\n")

print("Valores nulos por coluna:")
print(df.isnull().sum())

print("\nLinhas totalmente duplicadas:")
print(df[df.duplicated(keep=False)])

print("\nMoedas presentes:", df["moeda"].unique().tolist())

print("\nOperações com valor <= 0:", (df["valor"] <= 0).sum())

=== Diagnóstico de qualidade dos dados ===

Valores nulos por coluna:
id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

Linhas totalmente duplicadas:
        id cliente_id        data  valor moeda canal                   tipo          contraparte observacao
6  OP-0007    CLI-A-3  2026-03-05  17200   BRL   pix  transferencia_enviada  Epsilon Consultoria           
9  OP-0007    CLI-A-3  2026-03-05  17200   BRL   pix  transferencia_enviada  Epsilon Consultoria           

Moedas presentes: ['BRL', 'USD']

Operações com valor <= 0: 0


## 1. Limpeza dos dados

Três problemas encontrados no diagnóstico acima:

1. **Linha duplicada** — `OP-0007` aparece duas vezes com campos idênticos.
   Decisão: manter apenas a primeira ocorrência (`drop_duplicates`).

2. **Data nula** — `OP-0017` tem `data: null` (observação diz "data não capturada pelo sistema").
   Decisão: manter a operação (o valor e os demais campos são válidos), mas registrar
   o problema. Não inventar uma data fictícia — seria introduzir informação falsa.

3. **Moeda estrangeira** — `OP-0013` está em USD. Para que somas e comparações
   façam sentido, precisamos de um valor normalizado em BRL.
   Decisão: criar coluna `valor_brl` usando a taxa de câmbio fornecida no próprio JSON.

In [4]:
# 1. Remover duplicatas exatas
antes = len(df)
df = df.drop_duplicates()
print(f"Duplicatas removidas: {antes - len(df)} linha(s)")

# 2. Registrar operações com data nula (manter, não descartar)
nulos_data = df["data"].isnull().sum()
print(f"Operações com data nula (mantidas): {nulos_data}")

# 3. Normalizar valores para BRL
df["valor_brl"] = df.apply(
    lambda row: row["valor"] * taxa_cambio_usd_brl if row["moeda"] == "USD" else row["valor"],
    axis=1
)

print(f"\nOperações após limpeza: {len(df)}")
print(df[["id", "valor", "moeda", "valor_brl"]].to_string(index=False))

Duplicatas removidas: 1 linha(s)
Operações com data nula (mantidas): 1

Operações após limpeza: 19
     id  valor moeda  valor_brl
OP-0001  18100   BRL    18100.0
OP-0002  17300   BRL    17300.0
OP-0003  18800   BRL    18800.0
OP-0004   3300   BRL     3300.0
OP-0005  25900   BRL    25900.0
OP-0006  27000   BRL    27000.0
OP-0007  17200   BRL    17200.0
OP-0008  15200   BRL    15200.0
OP-0009  16100   BRL    16100.0
OP-0010   3800   BRL     3800.0
OP-0011   5100   BRL     5100.0
OP-0012   5800   BRL     5800.0
OP-0013  12000   USD    64800.0
OP-0014   2900   BRL     2900.0
OP-0015   7000   BRL     7000.0
OP-0016   2700   BRL     2700.0
OP-0017   4300   BRL     4300.0
OP-0018   8800   BRL     8800.0
OP-0019   1400   BRL     1400.0


## 2. Agregações e regras determinísticas

Antes das regras, duas agregações pedidas no enunciado:
- Volume total transacionado por cliente (em BRL)
- Quantidade de operações por canal

In [5]:
# Volume total por cliente (em BRL)
volume_por_cliente = df.groupby("cliente_id")["valor_brl"].sum().sort_values(ascending=False)
print("Volume total transacionado por cliente (BRL):")
print(volume_por_cliente.to_string())

print()

# Quantidade de operações por canal
ops_por_canal = df.groupby("canal")["id"].count().sort_values(ascending=False)
ops_por_canal.name = "n_operacoes"
print("Operações por canal:")
print(ops_por_canal.to_string())

Volume total transacionado por cliente (BRL):
cliente_id
CLI-A-4    79500.0
CLI-A-1    57500.0
CLI-A-2    52900.0
CLI-A-3    48500.0
CLI-A-5    16900.0
CLI-A-6    10200.0

Operações por canal:
canal
pix        8
ted        5
boleto     3
cartao     2
especie    1


### Regra 1 — Fracionamento

Sinalize o **cliente** que, em uma **mesma data**, realizou **3 ou mais operações**
cuja soma ultrapassa **R\$ 50.000**, sendo que **nenhuma operação isolada atinge R\$ 20.000**.

É o padrão clássico de *smurfing*: dividir um valor grande em transferências menores
para evitar disparar alertas automáticos.

In [6]:
# Regra 1 — Fracionamento
# Agrupa por cliente + data, calcula número de ops, soma e valor máximo
grupo_dia = df.groupby(["cliente_id", "data"]).agg(
    n_ops=("valor_brl", "count"),
    soma_dia=("valor_brl", "sum"),
    max_op=("valor_brl", "max"),
).reset_index()

# Aplica os três critérios simultaneamente
fracionamento = grupo_dia[
    (grupo_dia["n_ops"] >= 3) &
    (grupo_dia["soma_dia"] > 50_000) &
    (grupo_dia["max_op"] < 20_000)
]

clientes_regra1 = fracionamento["cliente_id"].unique().tolist()

# Flag no DataFrame original
df["flag_regra1_fracionamento"] = df["cliente_id"].isin(clientes_regra1)

print("Clientes sinalizados pela Regra 1 (fracionamento):", clientes_regra1)
print()
print("Detalhamento:")
print(fracionamento.to_string(index=False))

Clientes sinalizados pela Regra 1 (fracionamento): ['CLI-A-1']

Detalhamento:
cliente_id       data  n_ops  soma_dia  max_op
   CLI-A-1 2026-03-09      3   54200.0 18800.0


### Regra 2 — Valor atípico

Sinalize a **operação** cujo valor em BRL seja **superior a 5× a mediana** dos
valores daquele mesmo cliente. Aplica apenas a clientes com **4 ou mais operações**
(com poucas operações a mediana é instável e geraria falsos positivos).

In [7]:
# Regra 2 — Valor atípico (> 5× mediana do cliente, mínimo 4 ops)

# Calcular mediana por cliente
stats_cliente = df.groupby("cliente_id")["valor_brl"].agg(["median", "count"]).reset_index()
stats_cliente.columns = ["cliente_id", "mediana_brl", "n_ops"]

# Filtrar apenas clientes com 4+ operações
stats_4plus = stats_cliente[stats_cliente["n_ops"] >= 4].copy()
stats_4plus["limiar_5x"] = stats_4plus["mediana_brl"] * 5

print("Clientes elegíveis (4+ ops) e seus limiares:")
print(stats_4plus.to_string(index=False))
print()

# Merge e flag
df = df.merge(stats_4plus[["cliente_id", "mediana_brl", "limiar_5x"]], on="cliente_id", how="left")
df["flag_regra2_valor_atipico"] = df["valor_brl"] > df["limiar_5x"]

# Operações sinalizadas
flagged_r2 = df[df["flag_regra2_valor_atipico"]]
print("Operações sinalizadas pela Regra 2 (valor atípico):")
print(flagged_r2[["id", "cliente_id", "valor_brl", "mediana_brl", "limiar_5x"]].to_string(index=False))

Clientes elegíveis (4+ ops) e seus limiares:
cliente_id  mediana_brl  n_ops  limiar_5x
   CLI-A-1      17700.0      4    88500.0
   CLI-A-4       5450.0      4    27250.0
   CLI-A-5       3600.0      4    18000.0

Operações sinalizadas pela Regra 2 (valor atípico):
     id cliente_id  valor_brl  mediana_brl  limiar_5x
OP-0013    CLI-A-4    64800.0       5450.0    27250.0


## 3. Validação das regras

O enunciado pede que mostremos explicitamente que cada regra captura o caso correto
e **não** captura um caso parecido que não se enquadra.

In [8]:
# === Validação da Regra 1 (Fracionamento) ===
print("=== Validação Regra 1 ===\n")

# CASO POSITIVO: CLI-A-1 em 2026-03-09
print("✅ CASO POSITIVO — CLI-A-1 em 2026-03-09:")
caso_pos = df[(df["cliente_id"] == "CLI-A-1") & (df["data"] == "2026-03-09")]
print(caso_pos[["id", "valor_brl"]].to_string(index=False))
print(f"   Nº operações: {len(caso_pos)} (≥ 3 ✓)")
print(f"   Soma: R$ {caso_pos['valor_brl'].sum():,.2f} (> R$ 50.000 ✓)")
print(f"   Maior operação: R$ {caso_pos['valor_brl'].max():,.2f} (< R$ 20.000 ✓)")
print(f"   → Sinalizado: SIM\n")

# CASO NEGATIVO 1: CLI-A-3 em 2026-03-05 (soma < 50k)
print("❌ CASO NEGATIVO — CLI-A-3 em 2026-03-05 (soma insuficiente):")
caso_neg = df[(df["cliente_id"] == "CLI-A-3") & (df["data"] == "2026-03-05")]
print(caso_neg[["id", "valor_brl"]].to_string(index=False))
print(f"   Nº operações: {len(caso_neg)} (≥ 3 ✓)")
print(f"   Soma: R$ {caso_neg['valor_brl'].sum():,.2f} (> R$ 50.000 ✗ — abaixo do limiar)")
print(f"   Maior operação: R$ {caso_neg['valor_brl'].max():,.2f} (< R$ 20.000 ✓)")
print(f"   → Sinalizado: NÃO (correto — soma não ultrapassou R$ 50k)\n")

# CASO NEGATIVO 2: CLI-A-2 em 2026-03-14 (< 3 ops e ops > 20k)
print("❌ CASO NEGATIVO — CLI-A-2 em 2026-03-14 (poucas ops + valores altos):")
caso_neg2 = df[(df["cliente_id"] == "CLI-A-2") & (df["data"] == "2026-03-14")]
print(caso_neg2[["id", "valor_brl"]].to_string(index=False))
print(f"   Nº operações: {len(caso_neg2)} (≥ 3 ✗ — apenas 2)")
print(f"   Soma: R$ {caso_neg2['valor_brl'].sum():,.2f} (> R$ 50.000 ✓)")
print(f"   Maior operação: R$ {caso_neg2['valor_brl'].max():,.2f} (< R$ 20.000 ✗ — acima)")
print(f"   → Sinalizado: NÃO (correto — não é fracionamento, são operações grandes)")

print()
print("=" * 60)

# === Validação da Regra 2 (Valor atípico) ===
print("\n=== Validação Regra 2 ===\n")

# CASO POSITIVO: OP-0013 (CLI-A-4)
print("✅ CASO POSITIVO — OP-0013 (CLI-A-4):")
print(f"   Valor: R$ 64.800,00")
print(f"   Mediana do cliente: R$ 5.450,00")
print(f"   Limiar (5×): R$ 27.250,00")
print(f"   64.800 > 27.250 → Sinalizado: SIM\n")

# CASO NEGATIVO: OP-0015 (CLI-A-5, valor R$ 7.000)
print("❌ CASO NEGATIVO — OP-0015 (CLI-A-5):")
print(f"   Valor: R$ 7.000,00")
print(f"   Mediana do cliente: R$ 3.600,00")
print(f"   Limiar (5×): R$ 18.000,00")
print(f"   7.000 < 18.000 → Sinalizado: NÃO (correto — valor alto mas dentro do esperado)")

=== Validação Regra 1 ===

✅ CASO POSITIVO — CLI-A-1 em 2026-03-09:
     id  valor_brl
OP-0001    18100.0
OP-0002    17300.0
OP-0003    18800.0
   Nº operações: 3 (≥ 3 ✓)
   Soma: R$ 54,200.00 (> R$ 50.000 ✓)
   Maior operação: R$ 18,800.00 (< R$ 20.000 ✓)
   → Sinalizado: SIM

❌ CASO NEGATIVO — CLI-A-3 em 2026-03-05 (soma insuficiente):
     id  valor_brl
OP-0007    17200.0
OP-0008    15200.0
OP-0009    16100.0
   Nº operações: 3 (≥ 3 ✓)
   Soma: R$ 48,500.00 (> R$ 50.000 ✗ — abaixo do limiar)
   Maior operação: R$ 17,200.00 (< R$ 20.000 ✓)
   → Sinalizado: NÃO (correto — soma não ultrapassou R$ 50k)

❌ CASO NEGATIVO — CLI-A-2 em 2026-03-14 (poucas ops + valores altos):
     id  valor_brl
OP-0005    25900.0
OP-0006    27000.0
   Nº operações: 2 (≥ 3 ✗ — apenas 2)
   Soma: R$ 52,900.00 (> R$ 50.000 ✓)
   Maior operação: R$ 27,000.00 (< R$ 20.000 ✗ — acima)
   → Sinalizado: NÃO (correto — não é fracionamento, são operações grandes)


=== Validação Regra 2 ===

✅ CASO POSITIVO — OP-0013 